## **INIT**

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

## **Reading from bronze layer**

In [0]:
df_bronze =spark.table("transport_lakehouse.bronze.two_wheeled_plate_numbers_raw")
df_bronze.display()

## Columns organization and new column aliases

In [0]:

df_silver = df_bronze.select(
  F.col("mispar_rechev").cast("int").alias("car_num"),
  F.col("sug_delek_cd").cast("int").alias("fuel_type_cd"),
  F.col("sug_delek_nm").cast("string").alias("fuel_type_nm"),
  F.col("tozeret_cd").cast("int").alias("manufacturer_cd"),
  F.col("tozeret_nm").cast("string").alias("manufacturer_nm"),
  F.col("tozeret_eretz_nm").cast("string").alias("manufacturer_country_nm"),
  F.col("sug_rechev_cd").cast("int").alias("vehicle_type_cd"),
  F.col("sug_rechev_nm").cast("string").alias("vehicle_type_nm"),
  F.col("degem_nm").cast("string").alias("model_nm"),
  F.col("mishkal_kolel").cast("int").alias("total_weight"),
  F.col("baalut").cast("string").alias("ownership"),
  F.col("shnat_yitzur").cast("int").alias("prod_year"),
  F.col("moed_aliya_lakvish").cast("date").alias("road_entry_dt"),
)

df_silver.display()


## Null handling

In [0]:
df_silver = df_silver.withColumn(
    "road_entry_dt",
    F.coalesce(
        F.col("road_entry_dt"),
        F.to_date(
            F.concat(F.col("prod_year").cast("string"), F.lit("-01-01")),
            "yyyy-MM-dd"
        )
    )
)

df_silver = df_silver.withColumn(
    "manufacturer_country_nm",
    F.coalesce(
        F.col("manufacturer_country_nm"),
        F.when(
        F.col("manufacturer_cd") == 1678,
        "ויאטנם"
    ).otherwise(F.col("manufacturer_country_nm"))
        
    )
)

df_silver = df_silver.withColumn(
    "manufacturer_country_nm",
    F.coalesce(
        F.col("manufacturer_country_nm"),
        F.when(
        F.col("manufacturer_cd") == 64,
        "איטליה"
    ).otherwise(F.col("manufacturer_country_nm"))
        
    )
)
df_silver.display()

## String Fixes

In [0]:
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 892,
        "קואנג יאנג טאיוואן"
    ).otherwise(F.col("manufacturer_nm"))
)
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 560,
        "קוואסאקי תאילנד"
    ).otherwise(F.col("manufacturer_nm"))
)
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 560,
        "קי.טי.אמ אוסטריה"
    ).otherwise(F.col("manufacturer_nm"))
)
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 952,
        "דיילים ד.קוריאה"
    ).otherwise(F.col("manufacturer_nm"))
)
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 182,
        "בטהמוטור איטליה"
    ).otherwise(F.col("manufacturer_nm"))
)
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 1574,
        "אמ.בי.פי איטליה"
    ).otherwise(F.col("manufacturer_nm"))
)
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 1603,
        "אס ווי אמ טאיוואן"
    ).otherwise(F.col("manufacturer_nm"))
)
df_silver = df_silver.withColumn(
    "manufacturer_nm",
    F.when(
        F.col("manufacturer_cd") == 1678,
        "אס ווי אם ויאטנאם"
    ).otherwise(F.col("manufacturer_nm"))
)

df_silver.display()

## Trimming spaces

In [0]:
df_silver = df_silver.withColumn(
    "model_nm",
    F.trim(
        F.regexp_replace(F.col("model_nm"), r"\s+", " ")
    )
)

df_silver.display()

## Order df by vehicle type

In [0]:
df_silver_ordered = df_silver.orderBy(F.col("vehicle_type_cd").desc())

df_silver_ordered.display()

## Writing to silver layer

In [0]:
(
    df_silver_ordered.write 
        .mode("overwrite") 
        .format("delta") 
        .option("overwriteSchema", "true")
        .saveAsTable("transport_lakehouse.silver.silver_vehicles_two_wheeled")

)

spark.table("transport_lakehouse.silver.silver_vehicles_two_wheeled").display()